<a href="https://colab.research.google.com/github/Huii0529/Data-Management/blob/main/P167347_STQD6324_ASSIGNMENT1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Iris Dataset Classification with Spark MLlib: Model Implementation and Comparative Analysis**

This notebook demonstrates a complete machine learning workflow using Spark MLlib for classifying the Iris dataset. It covers data loading, preprocessing, model implementation (Decision Tree, Random Forest, Logistic Regression), hyperparameter tuning using cross-validation and grid search, evaluation of tuned models, and a comprehensive comparative analysis of their performance, strengths, and limitations.

**Workflow Overview:**
1.  **Data Loading**
2.  **Data Preprocessing**
3.  **Dataset Split**
4.  **Model Implementation (Initial)** Training and evaluating Decision Tree, Random Forest, and Logistic Regression models.
5.  **Model Tuning:** Optimizing hyperparameters using `CrossValidator` and `ParamGridBuilder` for each model.
6.  **Tuned Model Evaluation**
7.  **Comparative Analysis**

In [1]:
# Mount to Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [38]:
# Import the library
import pandas as pd
import numpy as np
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import Row
from pyspark.sql import functions
from pyspark.sql.functions import lit
from sklearn.model_selection import train_test_split

# Spark MLlib Imports
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

In [21]:
# Upload Iris.csv to Google Drive
from google.colab import files
files.upload()

Saving Iris.csv to Iris.csv


{'Iris.csv': b'Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species\n1,5.1,3.5,1.4,0.2,Iris-setosa\n2,4.9,3.0,1.4,0.2,Iris-setosa\n3,4.7,3.2,1.3,0.2,Iris-setosa\n4,4.6,3.1,1.5,0.2,Iris-setosa\n5,5.0,3.6,1.4,0.2,Iris-setosa\n6,5.4,3.9,1.7,0.4,Iris-setosa\n7,4.6,3.4,1.4,0.3,Iris-setosa\n8,5.0,3.4,1.5,0.2,Iris-setosa\n9,4.4,2.9,1.4,0.2,Iris-setosa\n10,4.9,3.1,1.5,0.1,Iris-setosa\n11,5.4,3.7,1.5,0.2,Iris-setosa\n12,4.8,3.4,1.6,0.2,Iris-setosa\n13,4.8,3.0,1.4,0.1,Iris-setosa\n14,4.3,3.0,1.1,0.1,Iris-setosa\n15,5.8,4.0,1.2,0.2,Iris-setosa\n16,5.7,4.4,1.5,0.4,Iris-setosa\n17,5.4,3.9,1.3,0.4,Iris-setosa\n18,5.1,3.5,1.4,0.3,Iris-setosa\n19,5.7,3.8,1.7,0.3,Iris-setosa\n20,5.1,3.8,1.5,0.3,Iris-setosa\n21,5.4,3.4,1.7,0.2,Iris-setosa\n22,5.1,3.7,1.5,0.4,Iris-setosa\n23,4.6,3.6,1.0,0.2,Iris-setosa\n24,5.1,3.3,1.7,0.5,Iris-setosa\n25,4.8,3.4,1.9,0.2,Iris-setosa\n26,5.0,3.0,1.6,0.2,Iris-setosa\n27,5.0,3.4,1.6,0.4,Iris-setosa\n28,5.2,3.5,1.5,0.2,Iris-setosa\n29,5.2,3.4,1.4,0.2,Iris-setosa\n

In [28]:
# Specified the delimiter: csv -> ','
df = pd.read_table('/content/Iris.csv', sep = ',')
df

,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa
...,...,...,...,...,...,...
145,146,6.7,3.0,5.2,2.3,Iris-virginica
146,147,6.3,2.5,5.0,1.9,Iris-virginica
147,148,6.5,3.0,5.2,2.0,Iris-virginica
148,149,6.2,3.4,5.4,2.3,Iris-virginica


## **Data Preprocessing**

This section prepares the loaded Iris dataset for machine learning. This involves converting the pandas DataFrame to a Spark DataFrame, performing feature vectorization, and indexing the categorical target variable (Species) into numerical labels, as required by Spark MLlib.

In [29]:
# Initialize SparkSession
spark = SparkSession.builder.getOrCreate()

In [31]:
# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Display the first 5 rows of the Spark DataFrame
spark_df.show(5)

# Print the schema of the Spark DataFrame
spark_df.printSchema()

+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  3|          4.7|         3.2|          1.3|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows
root
 |-- Id: long (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)



In [35]:
# Split the Spark DataFrame into training and testing sets
# 80% for training, 20% for testing
# The seed is for reproducibility
(spark_train_df, spark_test_df) = spark_df.randomSplit([0.8, 0.2], seed=42)

print("Number of rows for training:", spark_train_df.count())
print("\nFirst 5 rows of training dataset:")
spark_train_df.show(5)

print("\nNumber of rows for testing:", spark_test_df.count())
print("\nFirst 5 rows of testing dataset:")
spark_test_df.show(5)

Number of rows for training: 118

First 5 rows of training dataset:
+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+------------+-----------+
|  1|          5.1|         3.5|          1.4|         0.2|Iris-setosa|
|  2|          4.9|         3.0|          1.4|         0.2|Iris-setosa|
|  4|          4.6|         3.1|          1.5|         0.2|Iris-setosa|
|  5|          5.0|         3.6|          1.4|         0.2|Iris-setosa|
|  6|          5.4|         3.9|          1.7|         0.4|Iris-setosa|
+---+-------------+------------+-------------+------------+-----------+
only showing top 5 rows

Number of rows for testing: 32

First 5 rows of testing dataset:
+---+-------------+------------+-------------+------------+-----------+
| Id|SepalLengthCm|SepalWidthCm|PetalLengthCm|PetalWidthCm|    Species|
+---+-------------+------------+-------------+----

## **Decision Tree Implementation**

In [39]:
# Define feature columns (excluding 'Id' and 'Species')
feature_columns = [col for col in spark_df.columns if col not in ['Id', 'Species']]

# 1. Feature Vectorization: Combine features into a single vector column
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features")

# Transform the training and testing datasets
spark_train_df_assembled = assembler.transform(spark_train_df)
spark_test_df_assembled = assembler.transform(spark_test_df)

print("Schema after feature assembly on training data:")
spark_train_df_assembled.printSchema()
spark_train_df_assembled.select("features", "Species").show(5, truncate=False)

Schema after feature assembly on training data:
root
 |-- Id: long (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)
 |-- features: vector (nullable = true)

+-----------------+-----------+
|features         |Species    |
+-----------------+-----------+
|[5.1,3.5,1.4,0.2]|Iris-setosa|
|[4.9,3.0,1.4,0.2]|Iris-setosa|
|[4.6,3.1,1.5,0.2]|Iris-setosa|
|[5.0,3.6,1.4,0.2]|Iris-setosa|
|[5.4,3.9,1.7,0.4]|Iris-setosa|
+-----------------+-----------+
only showing top 5 rows


In [40]:
# 2. Label Indexing: Convert string labels to numerical indices
indexer = StringIndexer(inputCol="Species", outputCol="indexedLabel")

# Fit on the training data and transform both training and test data
indexer_model = indexer.fit(spark_train_df_assembled)
spark_train_df_indexed = indexer_model.transform(spark_train_df_assembled)
spark_test_df_indexed = indexer_model.transform(spark_test_df_assembled)

print("Schema after label indexing on training data:")
spark_train_df_indexed.printSchema()
spark_train_df_indexed.select("Species", "indexedLabel").show(5)

Schema after label indexing on training data:
root
 |-- Id: long (nullable = true)
 |-- SepalLengthCm: double (nullable = true)
 |-- SepalWidthCm: double (nullable = true)
 |-- PetalLengthCm: double (nullable = true)
 |-- PetalWidthCm: double (nullable = true)
 |-- Species: string (nullable = true)
 |-- features: vector (nullable = true)
 |-- indexedLabel: double (nullable = false)

+-----------+------------+
|    Species|indexedLabel|
+-----------+------------+
|Iris-setosa|         2.0|
|Iris-setosa|         2.0|
|Iris-setosa|         2.0|
|Iris-setosa|         2.0|
|Iris-setosa|         2.0|
+-----------+------------+
only showing top 5 rows


In [43]:
# 3. Model Training: Train a Decision Tree Classifier
dt = DecisionTreeClassifier(labelCol="indexedLabel", featuresCol="features", seed=42)

# Train model
dt_model = dt.fit(spark_train_df_indexed)

# Inspect the tree structure
print(dt_model.toDebugString)

DecisionTreeClassificationModel: uid=DecisionTreeClassifier_0d00a26d8932, depth=5, numNodes=17, numClasses=3, numFeatures=4
  If (feature 2 <= 2.45)
   Predict: 2.0
  Else (feature 2 > 2.45)
   If (feature 3 <= 1.75)
    If (feature 2 <= 4.95)
     If (feature 3 <= 1.65)
      Predict: 0.0
     Else (feature 3 > 1.65)
      Predict: 1.0
    Else (feature 2 > 4.95)
     If (feature 3 <= 1.55)
      Predict: 1.0
     Else (feature 3 > 1.55)
      If (feature 0 <= 6.75)
       Predict: 0.0
      Else (feature 0 > 6.75)
       Predict: 1.0
   Else (feature 3 > 1.75)
    If (feature 2 <= 4.85)
     If (feature 0 <= 5.95)
      Predict: 0.0
     Else (feature 0 > 5.95)
      Predict: 1.0
    Else (feature 2 > 4.85)
     Predict: 1.0



In [44]:
# Make predictions on the test data
predictions = dt_model.transform(spark_test_df_indexed)

print("Predictions on test data:")
predictions.select("features", "Species", "indexedLabel", "prediction", "rawPrediction", "probability").show(10, truncate=False)

Predictions on test data:
+-----------------+-----------+------------+----------+--------------+-------------+
|features         |Species    |indexedLabel|prediction|rawPrediction |probability  |
+-----------------+-----------+------------+----------+--------------+-------------+
|[4.7,3.2,1.3,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.6,3.4,1.4,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.4,2.9,1.4,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.3,3.0,1.1,0.1]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[5.1,3.8,1.5,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[5.1,3.3,1.7,0.5]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.7,3.2,1.6,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[5.0,3.2,1.2,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.8,3.0,1.4,0.3]|Iris-setosa|2.0     

In [45]:
# 4. Model Evaluation: Evaluate the model's performance
evaluator = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")

accuracy = evaluator.evaluate(predictions)
print(f"Test Accuracy = {accuracy}")

# Optionally, evaluate other metrics like F1-score
evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="f1")
f1_score = evaluator_f1.evaluate(predictions)
print(f"Test F1 Score = {f1_score}")

Test Accuracy = 1.0
Test F1 Score = 1.0


### Decision Tree Model Implementation on Iris Dataset

**Discussion:**

The Decision Tree Classifier was implemented using Spark MLlib to classify Iris species based on their sepal and petal measurements. The process involved several key steps:

1.  **Data Preparation:** The raw Iris dataset was first loaded into a pandas DataFrame, then converted into a Spark DataFrame (`spark_df`).
2.  **Train-Test Split:** The Spark DataFrame was split into training (`spark_train_df`) and testing (`spark_test_df`) sets with an 80/20 ratio to ensure the model's performance was evaluated on unseen data.
3.  **Feature Vectorization:** The numerical features (`SepalLengthCm`, `SepalWidthCm`, `PetalLengthCm`, `PetalWidthCm`) were combined into a single `features` vector column using `VectorAssembler`. This is a standard requirement for Spark MLlib models.
4.  **Label Indexing:** The categorical `Species` column (e.g., 'Iris-setosa') was transformed into a numerical `indexedLabel` column using `StringIndexer`, as machine learning models require numerical inputs for the target variable.
5.  **Model Training:** A `DecisionTreeClassifier` was initialized with the vectorized features and indexed labels, and then trained on the `spark_train_df_indexed` dataset.
6.  **Prediction:** The trained model (`dt_model`) was used to make predictions on the `spark_test_df_indexed` dataset, generating a `predictions` DataFrame that included the predicted labels and probabilities.
7.  **Model Evaluation:** The performance of the Decision Tree model was evaluated using `MulticlassClassificationEvaluator` to calculate accuracy and F1-score.

**Result Presentation:**

After training and evaluating the Decision Tree model on the Iris dataset, the following metrics were observed on the test set:

*   **Test Accuracy = 1.0**
*   **Test F1 Score = 1.0**

These results indicate that the Decision Tree model achieved perfect classification on the provided test dataset. This high performance suggests that the features in the Iris dataset are highly separable for the different species, and the Decision Tree effectively captured the underlying patterns. However, it's worth noting that perfect scores, especially on relatively small datasets, can sometimes indicate overfitting or a very straightforward classification problem. Further validation with more complex datasets or cross-validation could provide a more robust assessment.

## **Random Forest Implementation**

In [46]:
rf = RandomForestClassifier(labelCol="indexedLabel", featuresCol="features", seed=42)

# Train Random Forest model
rf_model = rf.fit(spark_train_df_indexed)
print("Random Forest Model trained.")

Random Forest Model trained.


In [47]:
# Make predictions on the test data using the Random Forest model
rf_predictions = rf_model.transform(spark_test_df_indexed)

print("Predictions on test data (Random Forest):")
rf_predictions.select("features", "Species", "indexedLabel", "prediction", "rawPrediction", "probability").show(10, truncate=False)

Predictions on test data (Random Forest):
+-----------------+-----------+------------+----------+--------------+---------------+
|features         |Species    |indexedLabel|prediction|rawPrediction |probability    |
+-----------------+-----------+------------+----------+--------------+---------------+
|[4.7,3.2,1.3,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.6,3.4,1.4,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.4,2.9,1.4,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.3,3.0,1.1,0.1]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[5.1,3.8,1.5,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[5.1,3.3,1.7,0.5]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.7,3.2,1.6,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[5.0,3.2,1.2,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|

In [48]:
# Evaluate Random Forest model performance (Accuracy)
rf_evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")

rf_accuracy = rf_evaluator_accuracy.evaluate(rf_predictions)
print(f"Random Forest Test Accuracy = {rf_accuracy}")

# Evaluate Random Forest model performance (F1-Score)
rf_evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="f1")

rf_f1_score = rf_evaluator_f1.evaluate(rf_predictions)
print(f"Random Forest Test F1 Score = {rf_f1_score}")

Random Forest Test Accuracy = 1.0
Random Forest Test F1 Score = 1.0


### Random Forest Model Implementation on Iris Dataset

**Discussion:**

The Random Forest Classifier was implemented as another robust machine learning model to classify Iris species. The process leveraged the same preprocessed data (feature vectorized and label indexed DataFrames) that were prepared for the Decision Tree model:

1.  **Model Training:** A `RandomForestClassifier` was initialized with the `indexedLabel` as the target and `features` as the input. The `seed=42` was used for reproducibility.
2.  **Prediction:** The trained `rf_model` then made predictions on the `spark_test_df_indexed` dataset, producing a `rf_predictions` DataFrame containing the predicted species.
3.  **Model Evaluation:** The performance was assessed using `MulticlassClassificationEvaluator` to calculate both accuracy and F1-score on the test set.

**Result Presentation:**

Upon executing the Random Forest model, the following performance metrics were observed on the test set:

*   **Random Forest Test Accuracy = 1.0**
*   **Random Forest Test F1 Score = 1.0**

Similar to the Decision Tree, the Random Forest model also achieved perfect accuracy and an F1-score of 1.0 on the Iris test dataset. This result is expected given the dataset's characteristics, where the classes are well-separated. Random Forests, being an ensemble method, generally offer higher robustness and can handle more complex relationships compared to single Decision Trees, though in this particular highly separable dataset, both models performed optimally.

## **Logistic Regression Implementation**

In [52]:
lr = LogisticRegression(labelCol="indexedLabel", featuresCol="features")

# Train Logistic Regression model
lr_model = lr.fit(spark_train_df_indexed)
print("Logistic Regression Model trained.")

Logistic Regression Model trained.


In [53]:
# Make predictions on the test data using the Logistic Regression model
lr_predictions = lr_model.transform(spark_test_df_indexed)

print("Predictions on test data (Logistic Regression):")
lr_predictions.select("features", "Species", "indexedLabel", "prediction", "rawPrediction", "probability").show(10, truncate=False)

Predictions on test data (Logistic Regression):
+-----------------+-----------+------------+----------+-----------------------------------------------------------+---------------------------------------------------+
|features         |Species    |indexedLabel|prediction|rawPrediction                                              |probability                                        |
+-----------------+-----------+------------+----------+-----------------------------------------------------------+---------------------------------------------------+
|[4.7,3.2,1.3,0.2]|Iris-setosa|2.0         |2.0       |[2.4723546577906585,-53.35758099801477,50.8852263402241]   |[9.430983287092775E-22,5.344708144298147E-46,1.0]  |
|[4.6,3.4,1.4,0.3]|Iris-setosa|2.0         |2.0       |[0.29015514658475716,-53.93669541972554,53.646540273140786]|[6.723824437002608E-24,1.8931908346696662E-47,1.0] |
|[4.4,2.9,1.4,0.2]|Iris-setosa|2.0         |2.0       |[3.91220273119305,-48.455574740086846,44.5433720088938]  

In [54]:
# Evaluate Logistic Regression model performance (Accuracy)
lr_evaluator_accuracy = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")

lr_accuracy = lr_evaluator_accuracy.evaluate(lr_predictions)
print(f"Logistic Regression Test Accuracy = {lr_accuracy}")

# Evaluate Logistic Regression model performance (F1-Score)
lr_evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="indexedLabel", predictionCol="prediction", metricName="f1")

lr_f1_score = lr_evaluator_f1.evaluate(lr_predictions)
print(f"Logistic Regression Test F1 Score = {lr_f1_score}")

Logistic Regression Test Accuracy = 1.0
Logistic Regression Test F1 Score = 1.0


### Logistic Regression Model Implementation on Iris Dataset

**Discussion:**

Logistic Regression, a linear model for classification, was applied to the Iris dataset. It uses the same preprocessed data (feature vectorized and label indexed DataFrames) as the previous models:

1.  **Model Training:** A `LogisticRegression` classifier was initialized, again specifying `indexedLabel` as the label column and `features` as the features column, with `seed=42` for reproducibility.
2.  **Prediction:** The trained `lr_model` was then used to make predictions on the `spark_test_df_indexed` dataset. The output `lr_predictions` DataFrame includes the predicted labels and probabilities.
3.  **Model Evaluation:** The model's performance was assessed using `MulticlassClassificationEvaluator` to calculate accuracy and F1-score on the test set.

**Result Presentation:**

For the Logistic Regression model, the following performance metrics were obtained on the test set:

*   **Logistic Regression Test Accuracy = 1.0**
*   **Logistic Regression Test F1 Score = 1.0**

Consistent with the Decision Tree and Random Forest models, Logistic Regression also achieved perfect accuracy and an F1-score of 1.0 on the Iris test dataset. This further reinforces the observation that the Iris dataset is highly separable for classification, and even a relatively simpler linear model like Logistic Regression can effectively distinguish between the different species.

## **Model Tuning: Decision Tree Classifier**

In [55]:
# 1. Define the parameter grid for Decision Tree
dt_paramGrid = ParamGridBuilder()\
    .addGrid(dt.maxDepth, [2, 5, 10]) \
    .addGrid(dt.minInfoGain, [0.0, 0.01, 0.1]) \
    .build()

# 2. Set up the CrossValidator
dt_crossval = CrossValidator(
    estimator=dt,
    estimatorParamMaps=dt_paramGrid,
    evaluator=MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy"),
    numFolds=3, # Use 3-fold cross-validation
    seed=42)

# 3. Train the tuned Decision Tree model
dt_cv_model = dt_crossval.fit(spark_train_df_indexed)

# Get the best model from CrossValidator
best_dt_model = dt_cv_model.bestModel

print("Tuned Decision Tree Model trained.")
print(f"Best Decision Tree maxDepth: {best_dt_model.getMaxDepth()}")
print(f"Best Decision Tree minInfoGain: {best_dt_model.getMinInfoGain()}")

Tuned Decision Tree Model trained.
Best Decision Tree maxDepth: 5
Best Decision Tree minInfoGain: 0.0


### Predictions with Tuned Decision Tree Model

In [56]:
# Make predictions on the test data using the best Decision Tree model
tuned_dt_predictions = best_dt_model.transform(spark_test_df_indexed)

print("Predictions on test data (Tuned Decision Tree):")
tuned_dt_predictions.select("features", "Species", "indexedLabel", "prediction", "rawPrediction", "probability").show(10, truncate=False)

Predictions on test data (Tuned Decision Tree):
+-----------------+-----------+------------+----------+--------------+-------------+
|features         |Species    |indexedLabel|prediction|rawPrediction |probability  |
+-----------------+-----------+------------+----------+--------------+-------------+
|[4.7,3.2,1.3,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.6,3.4,1.4,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.4,2.9,1.4,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.3,3.0,1.1,0.1]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[5.1,3.8,1.5,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[5.1,3.3,1.7,0.5]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.7,3.2,1.6,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[5.0,3.2,1.2,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,38.0]|[0.0,0.0,1.0]|
|[4.8,3.0,1.4,0.3

### Evaluation of Tuned Decision Tree Model

In [71]:
# Evaluate Tuned Decision Tree model performance

# Accuracy
dt_tuned_accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
dt_tuned_accuracy = dt_tuned_accuracy_evaluator.evaluate(tuned_dt_predictions)
print(f"Tuned Decision Tree Test Accuracy = {dt_tuned_accuracy}")

# F1-Score
dt_tuned_f1_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="f1")
dt_tuned_f1_score = dt_tuned_f1_evaluator.evaluate(tuned_dt_predictions)
print(f"Tuned Decision Tree Test F1 Score = {dt_tuned_f1_score}")

# Precision (weighted)
dt_tuned_precision_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedPrecision")
dt_tuned_precision = dt_tuned_precision_evaluator.evaluate(tuned_dt_predictions)
print(f"Tuned Decision Tree Test Weighted Precision = {dt_tuned_precision}")

# Recall (weighted)
dt_tuned_recall_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedRecall")
dt_tuned_recall = dt_tuned_recall_evaluator.evaluate(tuned_dt_predictions)
print(f"Tuned Decision Tree Test Weighted Recall = {dt_tuned_recall}")

Tuned Decision Tree Test Accuracy = 1.0
Tuned Decision Tree Test F1 Score = 1.0
Tuned Decision Tree Test Weighted Precision = 1.0
Tuned Decision Tree Test Weighted Recall = 1.0


### Tuned Decision Tree Model Implementation on Iris Dataset

**Discussion:**

To optimize the Decision Tree Classifier, hyperparameter tuning was applied using `ParamGridBuilder` and `CrossValidator`. This process helps in finding the best combination of hyperparameters that yield the best performance, reducing the risk of overfitting and improving generalization.

1.  **Parameter Grid:** A grid of hyperparameters was defined for `maxDepth` (controlling tree complexity) and `minInfoGain` (minimum information gain required for a split).
2.  **Cross-Validation:** A 3-fold cross-validation strategy was used with an `MulticlassClassificationEvaluator` set to optimize for `accuracy`. The `CrossValidator` trains multiple models across different parameter combinations and data folds, selecting the best model based on the evaluation metric.
3.  **Best Model Selection:** After the cross-validation process, the `best_dt_model` was obtained, representing the Decision Tree with the optimal hyperparameters found within the defined grid.
4.  **Prediction and Evaluation:** The tuned model was then used to make predictions on the test set, and its performance was thoroughly evaluated using accuracy, F1-score, weighted precision, and weighted recall.

**Result Presentation:**

After tuning, the best Decision Tree model had the following hyperparameters:
*   `Best Decision Tree maxDepth: 5`
*   `Best Decision Tree minInfoGain: 0.0`

The performance metrics of the tuned Decision Tree model on the test set are:

*   **Tuned Decision Tree Test Accuracy = 1.0**
*   **Tuned Decision Tree Test F1 Score = 1.0**
*   **Tuned Decision Tree Test Weighted Precision = 1.0**
*   **Tuned Decision Tree Test Weighted Recall = 1.0**

Even after tuning, the Decision Tree model continues to show perfect scores on the Iris dataset. This outcome reinforces the understanding that the Iris dataset is highly separable. Tuning confirmed that the model can achieve optimal performance with a relatively shallow tree (`maxDepth=5`) and without requiring a minimum information gain for splits, indicating clear boundaries between classes. In real-world scenarios, tuning would typically reveal more nuanced trade-offs in performance.

## **Model Tuning: Logistic Regression Classifier**

In [66]:
# 1. Define the parameter grid for Logistic Regression
lr_paramGrid = ParamGridBuilder()\
    .addGrid(lr.regParam, [0.01, 0.1, 0.5]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0]) \
    .build()

# 2. Set up the CrossValidator
lr_crossval = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_paramGrid,
    evaluator=MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy"),
    numFolds=3) # Use 3-fold cross-validation

# 3. Train the tuned Logistic Regression model
lr_cv_model = lr_crossval.fit(spark_train_df_indexed)

# Get the best model from CrossValidator
best_lr_model = lr_cv_model.bestModel

print("Tuned Logistic Regression Model trained.")
print(f"Best Logistic Regression regParam: {best_lr_model.getRegParam()}")
print(f"Best Logistic Regression elasticNetParam: {best_lr_model.getElasticNetParam()}")

Tuned Logistic Regression Model trained.
Best Logistic Regression regParam: 0.01
Best Logistic Regression elasticNetParam: 0.0


### Predictions with Tuned Logistic Regression Model

In [67]:
# Make predictions on the test data using the best Logistic Regression model
tuned_lr_predictions = best_lr_model.transform(spark_test_df_indexed)

print("Predictions on test data (Tuned Logistic Regression):")
tuned_lr_predictions.select("features", "Species", "indexedLabel", "prediction", "rawPrediction", "probability").show(10, truncate=False)

Predictions on test data (Tuned Logistic Regression):
+-----------------+-----------+------------+----------+----------------------------------------------------------+---------------------------------------------------------------+
|features         |Species    |indexedLabel|prediction|rawPrediction                                             |probability                                                    |
+-----------------+-----------+------------+----------+----------------------------------------------------------+---------------------------------------------------------------+
|[4.7,3.2,1.3,0.2]|Iris-setosa|2.0         |2.0       |[2.2566007149500233,-8.211690909889178,5.955090194939032] |[0.024162595454504925,6.867873624132509E-7,0.9758367177581326] |
|[4.6,3.4,1.4,0.3]|Iris-setosa|2.0         |2.0       |[1.9146715389110323,-8.146761590959077,6.232090052047924] |[0.013158791119491685,5.618121208987799E-7,0.9868406470683874] |
|[4.4,2.9,1.4,0.2]|Iris-setosa|2.0         |2.0    

### Evaluation of Tuned Logistic Regression Model

In [68]:
# Evaluate Tuned Logistic Regression model performance

# Accuracy
lr_tuned_accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
lr_tuned_accuracy = lr_tuned_accuracy_evaluator.evaluate(tuned_lr_predictions)
print(f"Tuned Logistic Regression Test Accuracy = {lr_tuned_accuracy}")

# F1-Score
lr_tuned_f1_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="f1")
lr_tuned_f1_score = lr_tuned_f1_evaluator.evaluate(tuned_lr_predictions)
print(f"Tuned Logistic Regression Test F1 Score = {lr_tuned_f1_score}")

# Precision (weighted)
lr_tuned_precision_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedPrecision")
lr_tuned_precision = lr_tuned_precision_evaluator.evaluate(tuned_lr_predictions)
print(f"Tuned Logistic Regression Test Weighted Precision = {lr_tuned_precision}")

# Recall (weighted)
lr_tuned_recall_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedRecall")
lr_tuned_recall = lr_tuned_recall_evaluator.evaluate(tuned_lr_predictions)
print(f"Tuned Logistic Regression Test Weighted Recall = {lr_tuned_recall}")

Tuned Logistic Regression Test Accuracy = 0.96875
Tuned Logistic Regression Test F1 Score = 0.968671679197995
Tuned Logistic Regression Test Weighted Precision = 0.9715909090909091
Tuned Logistic Regression Test Weighted Recall = 0.96875


### Tuned Logistic Regression Model Implementation on Iris Dataset

**Discussion:**

Logistic Regression was also subjected to hyperparameter tuning using `ParamGridBuilder` and `CrossValidator`. This helps in finding the optimal regularization parameters that can prevent overfitting and improve model generalization.

1.  **Parameter Grid:** A grid was defined for `regParam` (regularization parameter) and `elasticNetParam` (mixing parameter for L1 and L2 regularization).
2.  **Cross-Validation:** A 3-fold cross-validation was performed, optimizing for `accuracy` to select the best `LogisticRegressionModel`.
3.  **Best Model Selection:** The `best_lr_model` was identified, representing the Logistic Regression model with the best hyperparameter combination found during cross-validation.
4.  **Prediction and Evaluation:** The tuned model was then used to make predictions on the test set, and its performance was evaluated using accuracy, F1-score, weighted precision, and weighted recall.

**Result Presentation:**

After tuning, the best Logistic Regression model had the following hyperparameters:
*   `Best Logistic Regression regParam: 0.01`
*   `Best Logistic Regression elasticNetParam: 0.0`

The performance metrics of the tuned Logistic Regression model on the test set are:

*   **Tuned Logistic Regression Test Accuracy = 0.96875**
*   **Tuned Logistic Regression Test F1 Score = 0.968671679197995**
*   **Tuned Logistic Regression Test Weighted Precision = 0.9715909090909091**
*   **Tuned Logistic Regression Test Weighted Recall = 0.96875**

Unlike the Decision Tree and Random Forest, the tuned Logistic Regression model did not achieve perfect scores, but still performed exceptionally well with an accuracy of approximately 96.88%. This indicates that while the Iris dataset is highly separable, the linear nature of Logistic Regression might have slightly more difficulty in perfectly separating all data points compared to non-linear models like Decision Trees and Random Forests, especially at the boundaries between 'Iris-versicolor' and 'Iris-virginica'. The optimal parameters suggest a small amount of L2 regularization (`regParam=0.01`, `elasticNetParam=0.0`) was found to be best.

## **Model Tuning: Random Forest Classifier**

In [64]:
# 1. Define the parameter grid for Random Forest
rf_paramGrid = ParamGridBuilder()\
    .addGrid(rf.numTrees, [10, 20, 50]) \
    .addGrid(rf.maxDepth, [5, 10]) \
    .addGrid(rf.minInfoGain, [0.0, 0.01]) \
    .build()

# 2. Set up the CrossValidator
rf_crossval = CrossValidator(
    estimator=rf,
    estimatorParamMaps=rf_paramGrid,
    evaluator=MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy"),
    numFolds=3, # Use 3-fold cross-validation
    seed=42)

# 3. Train the tuned Random Forest model
rf_cv_model = rf_crossval.fit(spark_train_df_indexed)

# Get the best model from CrossValidator
best_rf_model = rf_cv_model.bestModel

print("Tuned Random Forest Model trained.")
print(f"Best Random Forest numTrees: {best_rf_model.getOrDefault(rf.numTrees)}")
print(f"Best Random Forest maxDepth: {best_rf_model.getOrDefault(rf.maxDepth)}")
print(f"Best Random Forest minInfoGain: {best_rf_model.getOrDefault(rf.minInfoGain)}")

Tuned Random Forest Model trained.
Best Random Forest numTrees: 20
Best Random Forest maxDepth: 5
Best Random Forest minInfoGain: 0.0


### Predictions with Tuned Random Forest Model

In [59]:
# Make predictions on the test data using the best Random Forest model
tuned_rf_predictions = best_rf_model.transform(spark_test_df_indexed)

print("Predictions on test data (Tuned Random Forest):")
tuned_rf_predictions.select("features", "Species", "indexedLabel", "prediction", "rawPrediction", "probability").show(10, truncate=False)

Predictions on test data (Tuned Random Forest):
+-----------------+-----------+------------+----------+--------------+---------------+
|features         |Species    |indexedLabel|prediction|rawPrediction |probability    |
+-----------------+-----------+------------+----------+--------------+---------------+
|[4.7,3.2,1.3,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.6,3.4,1.4,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.4,2.9,1.4,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.3,3.0,1.1,0.1]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[5.1,3.8,1.5,0.3]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[5.1,3.3,1.7,0.5]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[4.7,3.2,1.6,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0]  |
|[5.0,3.2,1.2,0.2]|Iris-setosa|2.0         |2.0       |[0.0,0.0,20.0]|[0.0,0.0,1.0

### Evaluation of Tuned Random Forest Model

In [60]:
# Evaluate Tuned Random Forest model performance

# Accuracy
rf_tuned_accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="accuracy")
rf_tuned_accuracy = rf_tuned_accuracy_evaluator.evaluate(tuned_rf_predictions)
print(f"Tuned Random Forest Test Accuracy = {rf_tuned_accuracy}")

# F1-Score
rf_tuned_f1_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="f1")
rf_tuned_f1_score = rf_tuned_f1_evaluator.evaluate(tuned_rf_predictions)
print(f"Tuned Random Forest Test F1 Score = {rf_tuned_f1_score}")

# Precision (weighted)
rf_tuned_precision_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedPrecision")
rf_tuned_precision = rf_tuned_precision_evaluator.evaluate(tuned_rf_predictions)
print(f"Tuned Random Forest Test Weighted Precision = {rf_tuned_precision}")

# Recall (weighted)
rf_tuned_recall_evaluator = MulticlassClassificationEvaluator(labelCol="indexedLabel", predictionCol="prediction", metricName="weightedRecall")
rf_tuned_recall = rf_tuned_recall_evaluator.evaluate(tuned_rf_predictions)
print(f"Tuned Random Forest Test Weighted Recall = {rf_tuned_recall}")

Tuned Random Forest Test Accuracy = 1.0
Tuned Random Forest Test F1 Score = 1.0
Tuned Random Forest Test Weighted Precision = 1.0
Tuned Random Forest Test Weighted Recall = 1.0


### Tuned Random Forest Model Implementation on Iris Dataset

**Discussion:**

Similar to the Decision Tree, the Random Forest Classifier underwent hyperparameter tuning using `ParamGridBuilder` and `CrossValidator` to find the optimal combination of parameters.

1.  **Parameter Grid:** A grid was defined for `numTrees` (number of trees in the forest), `maxDepth` (maximum depth of each tree), and `minInfoGain`.
2.  **Cross-Validation:** A 3-fold cross-validation was performed, optimizing for `accuracy` to select the best `RandomForestClassificationModel`.
3.  **Best Model Selection:** The `best_rf_model` was identified after the cross-validation process.
4.  **Prediction and Evaluation:** Predictions were generated on the test set using the best model, followed by a comprehensive evaluation including accuracy, F1-score, weighted precision, and weighted recall.

**Result Presentation:**

After tuning, the best Random Forest model had the following hyperparameters:
*   `Best Random Forest numTrees: 20`
*   `Best Random Forest maxDepth: 5`
*   `Best Random Forest minInfoGain: 0.0`

The performance metrics of the tuned Random Forest model on the test set are:

*   **Tuned Random Forest Test Accuracy = 1.0**
*   **Tuned Random Forest Test F1 Score = 1.0**
*   **Tuned Random Forest Test Weighted Precision = 1.0**
*   **Tuned Random Forest Test Weighted Recall = 1.0**

The tuned Random Forest model also achieved perfect scores, reaffirming the high separability of the Iris dataset. The optimal parameters suggest that a moderate number of trees with a limited depth are sufficient for this dataset, and further complexity does not improve performance. This result is consistent with the findings from the Decision Tree model, indicating that for this particular dataset, even complex models like Random Forest find clear and perfect decision boundaries.

## **Comprehensive Model Performance Comparison and Analysis**

This section provides a detailed comparative analysis of the three tuned classification models (Decision Tree, Random Forest, and Logistic Regression) on the Iris dataset.

### **Performance Metrics Comparison**

| Model                  | Accuracy | F1-Score | Weighted Precision | Weighted Recall |
|:-----------------------|:---------|:---------|:-------------------|:----------------|
| Tuned Decision Tree    | 1.0      | 1.0      | 1.0                | 1.0             |
| Tuned Random Forest    | 1.0      | 1.0      | 1.0                | 1.0             |
| Tuned Logistic Regression| 0.96875  | 0.96867  | 0.97159            | 0.96875         |

### **Strengths and Limitations of Each Model**

| Model                  | Strengths                                                                                                                              | Limitations                                                                                                                        |
|:-----------------------|:---------------------------------------------------------------------------------------------------------------------------------------|:-----------------------------------------------------------------------------------------------------------------------------------|
| Tuned Decision Tree    | * Easy to interpret<br>* Handles both numerical and categorical data<br>* Requires little data preprocessing.                           | * Prone to overfitting (especially without tuning)<br>* Can be unstable (small changes in data can lead to large changes in tree structure). |
| Tuned Random Forest    | * High accuracy, robust to overfitting (due to ensemble nature)<br>* Handles large datasets with many features<br>* Good for complex relationships. | * Less interpretable than single decision trees<br>* Can be computationally intensive and memory-heavy for very large forests.           |
| Tuned Logistic Regression| * Simple, efficient<br>* Good for linearly separable data<br>* Provides probability estimates<br>* Less prone to overfitting than complex models.   | * Assumes linearity<br>* May not perform well on complex, non-linear relationships<br>* Sensitive to outliers.                               |

## **Conclusion**

For the Iris dataset, both the **Tuned Decision Tree** and **Tuned Random Forest** models achieved perfect classification, demonstrating 100% accuracy and F1-scores across all evaluation metrics. This exceptional performance highlights the dataset's high separability, allowing these models to perfectly classify all instances. The **Tuned Logistic Regression** also performed very well, achieving an impressive accuracy of approximately 96.88%. While a single Decision Tree was sufficient for perfect classification due to the dataset's clear boundaries, the Random Forest, as an ensemble method, generally offers greater robustness and generalization capabilities for more complex real-world applications. For such scenarios, Random Forest would typically be preferred for its balance of accuracy and resistance to overfitting, whereas Logistic Regression serves as a strong, interpretable baseline, particularly for linearly separable data.